# Distill SFT: Qwen/Qwen2.5-1.5B-Instruct on DeepSeek-distilled, verify-filtered bricks + 2x2 eval
Accelerator **GPU T4** (fp16), Internet on. Attach the private dataset holding `brickexp_code.zip`.
Outputs in /kaggle/working: `sft/adapter`, `sft/train_log.json` (loss), `runs/eval_*.jsonl`, `runs/summary_*.json`, `TABLE_2x2.md`.

In [ ]:
import os, sys, glob, json, time, zipfile, subprocess, traceback
OUTD = "/kaggle/working" if os.path.exists("/kaggle") else "/content/work"
os.makedirs(OUTD, exist_ok=True); T0 = time.time()
def stage(n):
    print(f"=== [{(time.time()-T0)/60:6.1f} min] {n}", flush=True)
    open(f"{OUTD}/stage_{int(time.time()-T0):06d}_{n.replace(' ', '_')}.txt", "w").write(time.ctime())
def run_logged(cmd, tag):
    stage(tag)
    p = subprocess.run(f"set -o pipefail; ( {cmd} ) 2>&1 | tee {OUTD}/{tag}.log", shell=True, executable="/bin/bash",
                       capture_output=True, text=True)
    print(p.stdout[-4000:])
    if p.returncode != 0:
        open(f"{OUTD}/error.txt", "a").write(f"[{tag}] rc={p.returncode}\n" + p.stdout[-3000:])
        raise RuntimeError(f"{tag} failed rc={p.returncode}")
    return p
stage("pip")
r = subprocess.run("pip install -q vllm peft gurobipy datasets networkx bitsandbytes", shell=True, capture_output=True, text=True)
print(r.returncode, r.stderr[-1500:])
subprocess.run("pip uninstall -y -q torchao", shell=True)  # old torchao on Kaggle breaks recent peft import
print(subprocess.run("pip list 2>/dev/null | grep -iE '^(vllm|torch|transformers|peft) '", shell=True, capture_output=True, text=True).stdout)

In [ ]:
WORK = OUTD + "/RLtheLEGO"
zips = glob.glob("/kaggle/input/**/brickexp_code.zip", recursive=True) + glob.glob("/content/**/brickexp_code.zip", recursive=True)
if not os.path.exists(WORK + "/brickexp"):
    assert zips, "attach the dataset containing brickexp_code.zip"
    zipfile.ZipFile(zips[0]).extractall(WORK)
os.chdir(WORK); sys.path.insert(0, WORK)
print(sorted(os.listdir(WORK)))

In [ ]:
# Gurobi WLS license from Kaggle Secrets (never printed); written to /tmp, not the saved output dir
try:
    from kaggle_secrets import UserSecretsClient
    sc = UserSecretsClient()
    vals = {k: sc.get_secret(k) for k in ("GRB_WLSACCESSID", "GRB_WLSSECRET", "GRB_LICENSEID")}
    open("/tmp/gurobi.lic", "w").write("".join(f"{k[4:]}={v}\n" for k, v in vals.items())); del vals
    os.environ["GRB_LICENSE_FILE"] = "/tmp/gurobi.lic"
except Exception as e:
    print("no Gurobi secrets:", type(e).__name__)
from brickexp import checker
OK = checker.gurobi_available(); print("GUROBI SELF-CHECK:", "OK" if OK else "FAILED")
if "gurobi" == "gurobi" and not OK:
    raise checker.GurobiUnavailable("checker=gurobi but no working license; rebuild with --checker auto to fall back")
n = sum(1 for _ in open("data/distill/sft_distill.jsonl")); print("SFT examples:", n); assert n > 0

In [ ]:
# ---- LoRA SFT, fp16 on one T4 (subprocess so the GPU is free for vLLM afterwards) ----
TRAIN = (f"export CUDA_VISIBLE_DEVICES=0 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True; "
         f"python -m brickexp.train_sft --model Qwen/Qwen2.5-1.5B-Instruct --data_jsonl data/distill/sft_distill.jsonl --out {OUTD}/sft "
         f"--epochs 2 --bs 2 --grad_accum 8 --max_len 3072 --rank 16 --lr 2e-4 --max_steps -1")
pr = run_logged(TRAIN + " --probe_steps 5 --no_merge", "train_probe")   # memory probe before the full run
import re; m = re.search(r"PEAK_MEM_GB ([0-9.]+)", pr.stdout); PEAK = float(m.group(1)) if m else None
print("probe peak GB", PEAK); assert PEAK is not None and PEAK < 13.5, "would OOM on T4 (15 GB)"
run_logged(TRAIN, "train_full")
subprocess.run(f"rm -rf {OUTD}/sft/ckpt", shell=True)
L = json.load(open(f"{OUTD}/sft/train_log.json"))
print("loss:", [(h.get("step"), round(h["loss"], 4)) for h in L["log_history"] if "loss" in h])

In [ ]:
run_logged(f"python -m brickexp.eval_2x2 run --backend vllm --max_budget 3 --checker gurobi --limit 10 --time_budget_h 1.5 --out_dir {OUTD}/runs --model Qwen/Qwen2.5-1.5B-Instruct --tag base", "eval_base")

In [ ]:
run_logged(f"python -m brickexp.eval_2x2 run --backend vllm --max_budget 3 --checker gurobi --limit 10 --time_budget_h 1.5 --out_dir {OUTD}/runs --model {OUTD}/sft/merged --tag sft", "eval_sft")

In [ ]:
S = " ".join(f"{OUTD}/runs/summary_{t}.json" for t in ['base', 'sft'])
print(run_logged(f"python -m brickexp.eval_2x2 table {S} --out {OUTD}/TABLE_2x2.md", "table").stdout)
subprocess.run(f"rm -rf {OUTD}/sft/merged {OUTD}/RLtheLEGO", shell=True)  # keep adapter + logs only
stage("done")